# 02: Colab Pro and Unsloth Setup

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/02_colab_pro_and_unsloth_setup.ipynb)

Finetuning advice is only useful if the environment actually works. This notebook sets up a realistic Colab Pro workflow for open-source finetuning with Unsloth, then validates the stack with a first model load and dry run.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **Why environment discipline matters**
- Understand **Colab Pro expectations**
- Understand **Pre-install runtime probe**
- Understand **What the Unsloth stack includes**


## Learning goals

- understand why Colab runtime checks matter before any install
- inspect GPU, RAM, disk, and Python details from inside the notebook
- know which packages matter in the Unsloth stack and why
- recognize common environment failures and their most likely fixes
- complete a first FastLanguageModel.from_pretrained dry run on Colab-realistic hardware

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

ARTIFACT_DIR = Path("artifacts") / "notebook_02_colab_unsloth_setup"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def run_command(command):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    return {
        "command": command,
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
    }

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        headers = list(rows[0].keys())

    widths = []
    for header in headers:
        longest = max(len(str(row.get(header, ""))) for row in rows)
        widths.append(max(len(str(header)), longest))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(header, "")).ljust(widths[index]) for index, header in enumerate(headers)))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Why environment discipline matters

Open-source finetuning on Colab is practical, but it is still a real systems problem:

- runtimes change
- package versions drift
- CUDA support differs by wheel
- disk and RAM can quietly become the bottleneck

If setup is treated as magic, debugging becomes random. If setup is treated as a checklist, failure modes become manageable.

In [ ]:
runtime_rows = [
    {"item": "python_version", "value": sys.version.split()[0]},
    {"item": "platform", "value": platform.platform()},
    {"item": "executable", "value": sys.executable},
    {"item": "working_directory", "value": os.getcwd()},
]

print_table(runtime_rows, headers=["item", "value"])

## Colab Pro expectations

Colab Pro does not guarantee a single GPU type, so notebook design should be robust across the usual pool. The point is not to assume the best hardware. It is to detect what you got and pick a realistic recipe.

In [ ]:
hardware_expectations = [
    {
        "gpu": "T4",
        "typical_vram_gb": 16,
        "what_it_handles_well": "3B to 4B QLoRA, short to moderate contexts",
        "watch_out_for": "tight memory during longer contexts or larger adapters",
    },
    {
        "gpu": "L4",
        "typical_vram_gb": 24,
        "what_it_handles_well": "7B to 8B QLoRA more comfortably",
        "watch_out_for": "still not a license for dense tuning",
    },
    {
        "gpu": "A100",
        "typical_vram_gb": 40,
        "what_it_handles_well": "larger ablations and longer contexts",
        "watch_out_for": "cost discipline still matters",
    },
]

print_table(hardware_expectations, headers=["gpu", "typical_vram_gb", "what_it_handles_well", "watch_out_for"])

## Pre-install runtime probe

We can ask a few simple questions before installing anything:

- is there a visible GPU
- how much disk is available
- can the environment see standard CUDA diagnostics
- are we on a notebook-like Linux runtime

These checks save time when a runtime was opened without a GPU or with very little storage.

In [ ]:
disk = shutil.disk_usage(".")
commands = [
    run_command("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"),
    run_command("python --version"),
]

probe_rows = [
    {"check": "disk_total_gb", "value": round(disk.total / (1024 ** 3), 1)},
    {"check": "disk_free_gb", "value": round(disk.free / (1024 ** 3), 1)},
    {"check": "gpu_probe_returncode", "value": commands[0]["returncode"]},
    {"check": "gpu_probe_stdout", "value": commands[0]["stdout"][:120]},
    {"check": "python_probe", "value": commands[1]["stdout"]},
]

print_table(probe_rows, headers=["check", "value"])

## What the Unsloth stack includes

The stack in this course is intentionally narrow:

- unsloth for faster Colab-friendly loading and training
- datasets for data handling
- trl for supervised and preference training loops
- peft for adapter lifecycle
- accelerate and bitsandbytes for efficient runtime behavior

Keeping the stack small makes it easier to debug.

In [ ]:
package_roles = [
    {"package": "unsloth", "purpose": "fast model loading and training helpers"},
    {"package": "datasets", "purpose": "dataset loading and preprocessing"},
    {"package": "trl", "purpose": "SFT and alignment trainers"},
    {"package": "peft", "purpose": "LoRA and adapter management"},
    {"package": "accelerate", "purpose": "device placement and runtime utilities"},
    {"package": "bitsandbytes", "purpose": "4-bit and 8-bit quantization kernels"},
]

print_table(package_roles, headers=["package", "purpose"])

## Common environment problems before the first model load

A few failures show up repeatedly:

- no GPU runtime was selected
- the runtime reset and packages disappeared
- torch sees CUDA but the wheel mix is inconsistent
- disk filled up while caching weights
- you requested a model too large for the current GPU

The goal is not to memorize every error message. The goal is to spot the category quickly.

In [ ]:
common_failures = [
    {
        "symptom": "nvidia-smi not found or empty",
        "likely_cause": "CPU runtime or detached GPU",
        "first_fix": "switch runtime to GPU and reconnect",
    },
    {
        "symptom": "CUDA out of memory during load",
        "likely_cause": "model or context choice is too large",
        "first_fix": "drop to a smaller model or shorter sequence length",
    },
    {
        "symptom": "ImportError around bitsandbytes or CUDA",
        "likely_cause": "wheel mismatch after partial installs",
        "first_fix": "restart runtime and reinstall cleanly",
    },
    {
        "symptom": "Model download stalls or fails",
        "likely_cause": "network or cache pressure",
        "first_fix": "remount drive and retry after checking free disk",
    },
]

print_table(common_failures, headers=["symptom", "likely_cause", "first_fix"])

## Sanity checks before the heavy setup cell

These checks do not guarantee success, but they tell us whether proceeding makes sense.

In [ ]:
preflight = {
    "python_ok": sys.version_info >= (3, 10),
    "disk_free_gb": round(disk.free / (1024 ** 3), 1),
    "looks_like_linux": sys.platform.startswith("linux"),
    "gpu_visible": commands[0]["returncode"] == 0 and bool(commands[0]["stdout"]),
}

for key, value in preflight.items():
    print(key, "=", value)

if not preflight["gpu_visible"]:
    print()
    print("Warning: a GPU was not detected. In Colab, switch Runtime -> Change runtime type -> GPU.")
if preflight["disk_free_gb"] < 10:
    print()
    print("Warning: free disk is low for model caching and notebook artifacts.")

## Full setup with the shared generator cell

The next cell is the shared course setup:

- upgrades pip
- installs the Unsloth stack
- mounts Google Drive
- loads a 4-bit instruct model
- attaches a default LoRA adapter recipe

That means the environment is not just installed. It is validated by actually loading the stack.

In [ ]:
# --- Google Colab Pro Fine-Tuning Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Post-setup verification

A successful install is not the same as a healthy runtime. We still want to inspect device properties, memory state, and package versions.

In [ ]:
import importlib

device_rows = [
    {"item": "cuda_available", "value": torch.cuda.is_available()},
    {"item": "gpu_name", "value": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"},
    {"item": "bf16_supported", "value": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False},
    {"item": "allocated_gb", "value": round(torch.cuda.memory_allocated() / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0},
    {"item": "reserved_gb", "value": round(torch.cuda.memory_reserved() / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0},
]

print_table(device_rows, headers=["item", "value"])
print()

for package_name in ["torch", "datasets", "trl", "peft", "accelerate", "bitsandbytes", "unsloth"]:
    module = importlib.import_module(package_name)
    version = getattr(module, "__version__", "unknown")
    print(package_name, "version:", version)

## First dry run with FastLanguageModel.from_pretrained

The shared setup already loaded the model. Now we do the simplest meaningful test:

1. format a short chat prompt
2. generate a tiny response
3. confirm the tokenizer and model cooperate end to end

This is the moment where installation becomes a usable notebook environment.

In [ ]:
FastLanguageModel.for_inference(model)

def quick_chat(user_text, max_new_tokens=64):
    messages = [{"role": "user", "content": user_text}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

dry_run_text = quick_chat("In two sentences, explain why QLoRA is useful on Colab.")
print(dry_run_text)

## Inspect the loaded model and tokenizer

The first dry run answers whether the stack works. The next question is what exactly we loaded.

In [ ]:
model_rows = [
    {"item": "model_name", "value": MODEL_NAME},
    {"item": "max_seq_length", "value": MAX_SEQ_LENGTH},
    {"item": "load_in_4bit", "value": LOAD_IN_4BIT},
    {"item": "tokenizer_vocab_size", "value": len(tokenizer)},
    {"item": "pad_token_id", "value": tokenizer.pad_token_id},
    {"item": "eos_token_id", "value": tokenizer.eos_token_id},
]

print_table(model_rows, headers=["item", "value"])
print()
print("Model class:", type(model).__name__)
print("Tokenizer class:", type(tokenizer).__name__)

## A simple symptom triage helper

When something fails, you want your first guess to be informed, not random. The helper below maps rough symptoms to likely next checks.

In [ ]:
def triage(symptom):
    symptom = symptom.lower()
    if "out of memory" in symptom or "oom" in symptom:
        return [
            "Reduce model size or sequence length.",
            "Restart the runtime to clear fragmented memory.",
            "Prefer QLoRA before considering denser recipes.",
        ]
    if "bitsandbytes" in symptom or "cuda" in symptom:
        return [
            "Restart runtime and reinstall the stack cleanly.",
            "Verify that a GPU runtime is attached.",
            "Avoid mixing old cached wheels from prior sessions.",
        ]
    if "download" in symptom or "timeout" in symptom:
        return [
            "Check free disk and Google Drive mount state.",
            "Retry the model load after reconnecting the runtime.",
            "Use a smaller checkpoint if bandwidth is unstable.",
        ]
    if "no module named" in symptom:
        return [
            "Re-run the shared setup cell.",
            "Confirm the runtime did not reset after install.",
            "Restart and reinstall if imports are half-broken.",
        ]
    return [
        "Read the exact traceback carefully.",
        "Classify whether this is environment, memory, or code logic.",
        "Minimize the failing example before changing packages.",
    ]

for prompt in [
    "CUDA out of memory while loading the model",
    "No module named unsloth",
    "bitsandbytes CUDA setup failed",
]:
    print("Symptom:", prompt)
    for line in triage(prompt):
        print("-", line)
    print()

## Cache paths and rerun behavior

Colab sessions are temporary. Google Drive caching matters because repeated model downloads waste time and sometimes fail halfway through. This course therefore stores model weights and training outputs in Drive-backed locations.

In [ ]:
cache_rows = [
    {"path_name": "CACHE_DIR", "path_value": CACHE_DIR},
    {"path_name": "OUTPUT_DIR", "path_value": OUTPUT_DIR},
]

print_table(cache_rows, headers=["path_name", "path_value"])
print()

for folder in [CACHE_DIR, OUTPUT_DIR]:
    path_obj = Path(folder)
    print("Exists:", path_obj.exists(), "|", path_obj)
    if path_obj.exists():
        try:
            entries = list(path_obj.iterdir())
            print("Top-level entries:", len(entries))
        except Exception as exc:
            print("Could not inspect directory:", exc)
    print()

## Takeaways

- always detect the runtime before assuming a recipe will fit
- treat installation and first model load as part of one validation loop
- Unsloth plus QLoRA is the default practical path for this course
- common failures usually fall into GPU visibility, package mismatch, disk pressure, or model size mismatch

In the next notebook, we will use these environment assumptions to choose models and budget VRAM realistically.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** modify a hyperparameter and track its effect on loss curves. Document what changes and why.

**Exercise 2 — Build:** prepare a dataset from a new domain using the techniques shown. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** compare the finetuned model against the base model on 5 test prompts.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the finetuning/ directory